![mental_health_banner_1774025040503.jpg](../mental_health_banner_1774025040503.jpg "mental_health_banner_1774025040503.jpg")

### Ingestion de Datos

Ingesta de Datos US Temperature

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "health_db_catalog")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "saafadproject")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/us_temperature_by_state.csv"

In [0]:
us_temperature_schema = StructType([
    StructField("state", StringType(), True),
    StructField("month", IntegerType(), True),
    StructField("year", IntegerType(), True),
    StructField("average_temp", DecimalType(3,1), True),
    StructField("id", IntegerType(), True)    
])

In [0]:
df_us_temperature = (
    spark
    .read
    .schema(us_temperature_schema)
    .csv(ruta)
)

In [0]:
df_us_temperature_final = (df_us_temperature.select(col("id").alias("state_id"),
                                                    col("state"),
                                                    col("month"),
                                                    col("year"),
                                                    col("average_temp").alias("avg_temp")
                                                   ).withColumn("ingestion_date", current_timestamp())
)

In [0]:
df_us_temperature_final.coalesce(1).write.mode("overwrite").option("mergeSchema", "true").insertInto(f"{catalogo}.{esquema}.temperature")